In [18]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [19]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
DATASET_PATH = "/content/drive/MyDrive/Datasets/cotton"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Datasets/cotton/model_cotton_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model
# ---------------------------------------------------------
def build_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("GPU Available:", tf.config.list_physical_devices('GPU'))

    print("Loading dataset...")
    df = load_dataset(DATASET_PATH)

    # 🔥 Manual 80/20 split
    train_df, val_df = train_test_split(
        df,
        train_size=0.8,
        stratify=df["Label"],
        random_state=42
    )

    # 🔥 Strong Augmentation (small dataset)
    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=40,
        zoom_range=0.4,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.3,
        brightness_range=[0.5, 1.5],
        horizontal_flip=True,
        fill_mode="nearest"
    )

    val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = val_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Classes:", train_images.class_indices)

    model = build_model(len(train_images.class_indices))

    model.compile(
        optimizer=Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Training on GPU 🚀 ...")
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    # Save Accuracy Plot
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/cotton/accuracy.png")
    plt.close()

    # Save Loss Plot
    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/cotton/loss.png")
    plt.close()


train()


GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Loading dataset...
Found 1367 validated image filenames belonging to 4 classes.
Found 342 validated image filenames belonging to 4 classes.
Classes: {'bacterial_blight': 0, 'curl_virus': 1, 'fussarium_wilt': 2, 'healthy': 3}
Training on GPU 🚀 ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 465s 10s/step - accuracy: 0.3704 - loss: 1.4094 - val_accuracy: 0.7515 - val_loss: 0.8574
Epoch 2/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 30s 690ms/step - accuracy: 0.7281 - loss: 0.7902 - val_accuracy: 0.8567 - val_loss: 0.5545
Epoch 3/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 29s 684ms/step - accuracy: 0.8270 - loss: 0.5371 - val_accuracy: 0.8889 - val_loss: 0.3829
Epoch 4/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 29s 679ms/step - accuracy: 0.8433 - loss: 0.4759 - val_accuracy: 0.9035 - val_loss: 0.2919
Epoch 5/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 30s 692ms/step - accuracy: 0.8775 - loss: 0.3624 - val_accuracy: 0.9152 - val_loss: 0.2447
Epoch 6/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 30s 695ms/step - accuracy: 0.8845 - loss: 0.3081 - val_accuracy: 0.9211 - val_loss: 0.2144
Epoch 7/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 31s 710ms/step - accuracy: 0.9104 - loss: 0.2597 - val_accuracy: 0.9503 - val_loss: 0.1779
Epoch 8/10
43/43 ━━━━━━━━━━━━━━━━━━━━ 29s 680ms/step - accuracy: 0.9304 - loss: 0.2027 - val_accur